# 02. マッピング & カウントトリミング済みFASTQをhg38ゲノムにマッピングし、遺伝子ごとのリードカウント行列を生成します。**カーネル**: RNA-seq (Python) / rnaseq_env  **必要ツール**: STAR, featureCounts (Subread), samtools  **入力**: `trimmed/` 内のトリミング済みFASTQ  **出力**: `mapped/` にBAM, `results/count_matrix.csv`

## 設定

In [ ]:
import os, glob, subprocess, re, logging
import pandas as pd

# === 設定 ===
TRIMMED_DIR = "trimmed"
MAPPED_DIR = "mapped"
RESULTS_DIR = "results"
THREADS = 8

# ★ ユーザーが編集する箇所 ★
GENOME_DIR = "reference/hg38/star_index"   # STARインデックスのパス
GTF_FILE = "reference/hg38/gencode.v44.primary_assembly.annotation.gtf"  # GTFファイルのパス

# STARパラメータ
SJDB_OVERHANG = 149         # リード長 - 1 (150bpリードの場合は149)

# featureCountsパラメータ
STRANDEDNESS = 0            # 0=unstranded, 1=stranded, 2=reversely stranded

for d in [MAPPED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ロギング設定
LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(LOG_DIR, "02_mapping_and_counting.log")),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

def run_cmd(cmd, step_name=""):
    """外部コマンドを実行し、失敗時にエラーを投げる"""
    logger.info(f"実行: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        logger.error(f"{step_name} 失敗 (exit code {result.returncode})")
        logger.error(f"stderr: {result.stderr}")
        raise RuntimeError(f"{step_name} が失敗しました:\n{result.stderr}")
    logger.info(f"{step_name} 完了")
    return result

# パスの確認
assert os.path.isdir(GENOME_DIR), f"STARインデックスが見つかりません: {GENOME_DIR}"
assert os.path.isfile(GTF_FILE), f"GTFファイルが見つかりません: {GTF_FILE}"
print("設定完了 - リファレンスファイル確認OK")

## サンプルメタデータの読み込み

In [ ]:
metadata = pd.read_csv("sample_metadata.csv")
print(f"サンプル数: {len(metadata)}")
print(metadata.to_string(index=False))

## Step 1: STAR マッピング

In [ ]:
%%time
for _, row in metadata.iterrows():
    sample_id = row["sample_id"]
    
    # トリミング後のファイル名パターンを検出
    r1_pattern = os.path.join(TRIMMED_DIR, f"*{sample_id}*_val_1.fq.gz")
    r2_pattern = os.path.join(TRIMMED_DIR, f"*{sample_id}*_val_2.fq.gz")
    
    r1_files = glob.glob(r1_pattern)
    r2_files = glob.glob(r2_pattern)
    
    if not r1_files or not r2_files:
        # 元のファイル名ベースで探す
        r1_base = os.path.basename(row["fastq_r1"]).replace(".fastq.gz", "_val_1.fq.gz")
        r2_base = os.path.basename(row["fastq_r2"]).replace(".fastq.gz", "_val_2.fq.gz")
        r1_files = [os.path.join(TRIMMED_DIR, r1_base)]
        r2_files = [os.path.join(TRIMMED_DIR, r2_base)]
    
    r1 = r1_files[0]
    r2 = r2_files[0]
    
    if not os.path.isfile(r1):
        raise FileNotFoundError(f"トリミング済みR1が見つかりません: {r1}")
    if not os.path.isfile(r2):
        raise FileNotFoundError(f"トリミング済みR2が見つかりません: {r2}")
    
    prefix = os.path.join(MAPPED_DIR, f"{sample_id}_")
    
    cmd = (
        f"STAR --runThreadN {THREADS} "
        f"--genomeDir {GENOME_DIR} "
        f"--readFilesIn {r1} {r2} "
        f"--readFilesCommand zcat "
        f"--outFileNamePrefix {prefix} "
        f"--outSAMtype BAM SortedByCoordinate "
        f"--outSAMattributes NH HI AS NM MD "
        f"--sjdbOverhang {SJDB_OVERHANG} "
        f"--quantMode GeneCounts "
        f"--outBAMsortingThreadN {THREADS}"
    )
    
    print(f"マッピング中: {sample_id}")
    run_cmd(cmd, f"STAR ({sample_id})")
    
    # BAMインデックス作成
    bam = f"{prefix}Aligned.sortedByCoord.out.bam"
    if not os.path.isfile(bam):
        raise FileNotFoundError(f"BAMファイルが生成されませんでした: {bam}")
    run_cmd(f"samtools index -@ {THREADS} {bam}", f"samtools index ({sample_id})")

print("\nSTAR マッピング完了")

## Step 2: マッピング統計

In [ ]:
# STARのログから統計を取得
stats = []
for _, row in metadata.iterrows():
    log_file = os.path.join(MAPPED_DIR, f"{row['sample_id']}_Log.final.out")
    if os.path.exists(log_file):
        info = {"sample_id": row["sample_id"]}
        with open(log_file) as f:
            for line in f:
                m_unique = re.search(r"Uniquely mapped reads %\s*\|\s*(.+)", line)
                m_total = re.search(r"Number of input reads\s*\|\s*(.+)", line)
                if m_unique:
                    info["unique_map_rate"] = m_unique.group(1).strip()
                elif m_total:
                    info["total_reads"] = m_total.group(1).strip()
        stats.append(info)
    else:
        logger.warning(f"STARログが見つかりません: {log_file}")

stats_df = pd.DataFrame(stats)
print("マッピング統計:")
print(stats_df.to_string(index=False))

## Step 3: featureCounts でカウント行列生成

In [ ]:
%%time
bam_files = []
for _, row in metadata.iterrows():
    bam = os.path.join(MAPPED_DIR, f"{row['sample_id']}_Aligned.sortedByCoord.out.bam")
    assert os.path.exists(bam), f"BAMファイルが見つかりません: {bam}"
    bam_files.append(bam)

bam_str = " ".join(bam_files)
output_file = os.path.join(RESULTS_DIR, "featureCounts_output.txt")

cmd = (
    f"featureCounts -a {GTF_FILE} "
    f"-o {output_file} "
    f"-p --countReadPairs "
    f"-T {THREADS} "
    f"-s {STRANDEDNESS} "
    f"-B -C "
    f"{bam_str}"
)

print("featureCounts 実行中...")
run_cmd(cmd, "featureCounts")
print("featureCounts 完了")

## Step 4: カウント行列の整形

In [ ]:
# featureCountsの出力を読み込み、整形
fc_output = pd.read_csv(
    os.path.join(RESULTS_DIR, "featureCounts_output.txt"),
    sep="\t", comment="#"
)

# 遺伝子IDとカウントカラムのみ抽出
gene_col = "Geneid"
count_cols = [c for c in fc_output.columns if c.endswith(".bam")]

count_matrix = fc_output[[gene_col] + count_cols].copy()
count_matrix = count_matrix.set_index(gene_col)

# カラム名をサンプルIDに変換（パスからsample_idを正確に抽出）
new_names = {}
for col in count_cols:
    basename = os.path.basename(col)
    matched = False
    for _, row in metadata.iterrows():
        expected_prefix = f"{row['sample_id']}_Aligned"
        if basename.startswith(expected_prefix):
            new_names[col] = row["sample_id"]
            matched = True
            break
    if not matched:
        logger.warning(f"カラム '{col}' に一致するサンプルIDが見つかりません")

count_matrix = count_matrix.rename(columns=new_names)

# サンプル順をメタデータに合わせる
count_matrix = count_matrix[metadata["sample_id"].tolist()]

# CSV出力
count_matrix.to_csv(os.path.join(RESULTS_DIR, "count_matrix.csv"))

print(f"カウント行列: {count_matrix.shape[0]} genes x {count_matrix.shape[1]} samples")
print(f"出力: {RESULTS_DIR}/count_matrix.csv")
print("\nサンプル別の総リードカウント:")
print(count_matrix.sum().to_string())

## Step 5: Gene ID → Gene Name 変換テーブルの生成

In [ ]:
import re

# GTFからgene_id → gene_name の対応表を作成
print("GTFから遺伝子名を抽出中...")
gene_id_to_name = {}
with open(GTF_FILE) as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 9 or parts[2] != 'gene':
            continue
        attrs = parts[8]
        gid_m = re.search(r'gene_id "([^"]+)"', attrs)
        gname_m = re.search(r'gene_name "([^"]+)"', attrs)
        if gid_m and gname_m:
            # バージョン番号を除去 (ENSG00000000003.15 → ENSG00000000003)
            gene_id_stripped = gid_m.group(1).split('.')[0]
            gene_id_to_name[gene_id_stripped] = gname_m.group(1)

print(f"変換テーブルのエントリ数: {len(gene_id_to_name)}")

# 変換テーブルをCSVとして保存
name_df = pd.DataFrame([
    {"gene_id": k, "gene_name": v} for k, v in gene_id_to_name.items()
])
name_df.to_csv(os.path.join(RESULTS_DIR, "gene_id_to_name.csv"), index=False)
print(f"出力: {RESULTS_DIR}/gene_id_to_name.csv")

# カウント行列にgene_nameを付与してannotated版を保存
count_matrix_reload = pd.read_csv(os.path.join(RESULTS_DIR, "count_matrix.csv"), index_col=0)
# インデックス(ENSG ID)のバージョン番号を除去
count_matrix_reload.index = count_matrix_reload.index.str.split('.').str[0]
count_matrix_reload.index.name = "gene_id"
count_matrix_reload.insert(0, "gene_name",
    count_matrix_reload.index.map(lambda x: gene_id_to_name.get(x, x)))
count_matrix_reload.to_csv(os.path.join(RESULTS_DIR, "count_matrix_annotated.csv"))
print(f"出力: {RESULTS_DIR}/count_matrix_annotated.csv")

# カウント行列のgene_idもバージョン除去して上書き保存
count_matrix_clean = count_matrix_reload.drop(columns=["gene_name"])
count_matrix_clean.to_csv(os.path.join(RESULTS_DIR, "count_matrix.csv"))
print("count_matrix.csv のgene IDからバージョン番号を除去しました")


## 完了- `results/count_matrix.csv` が生成されました- 次のノートブック `03_DEG_Analysis.ipynb` に進んでください- Rカーネルに切り替えることを忘れないでください